In [2]:
import folium
from folium.plugins import MarkerCluster
import json
import pandas as pd
import html as html_module
import re
from math import radians, cos, sin, sqrt
from pathlib import Path
from IPython.display import display, HTML

# =========================================================
# CONFIG
# =========================================================
BASE_DIR     = Path(".")
TEXTS_DIR    = BASE_DIR / "data_texts"
FR_PATH      = TEXTS_DIR / "20000lieues_fr.txt"
JSON_PATH    = BASE_DIR / "Points-escales-chapitres.json"
OUTPUT_HTML  = BASE_DIR / "nautilus_map_v16.html"

for p in [TEXTS_DIR, FR_PATH, JSON_PATH]:
    assert p.exists(), f"Introuvable : {p.resolve()}"
    print(f"✅ {p.resolve()}")

fr_text_full = FR_PATH.read_text(encoding="utf-8", errors="ignore")
fr_text_full = fr_text_full.replace("\r\n", "\n").replace("\r", "\n")
print(f"📖 FR : {len(fr_text_full):,} caractères")

with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)
print(f"🗺️  {len(data)} escales chargées")

# =========================================================
# UTILITAIRES
# =========================================================
def clean_text(txt):
    if txt is None: return ""
    txt = str(txt)
    txt = txt.replace("\u00a0", " ").replace("\ufeff", "")
    txt = re.sub(r"\r\n?", "\n", txt)
    txt = re.sub(r"[ \t]+", " ", txt)
    txt = re.sub(r"\n{3,}", "\n\n", txt)
    return txt.strip()

def get_lang(value, lang="fr"):
    if isinstance(value, dict):
        v = value.get(lang) or value.get("fr") or ""
        return clean_text(str(v)) if v else ""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    return clean_text(str(value))

def distance_from_center(lat):
    R = 6371e3
    lat_rad = radians(lat)
    return int(sqrt((R * cos(lat_rad))**2 + (R * sin(lat_rad))**2))

print("✅ Utilitaires chargés")

# =========================================================
# ANCRES — extraites manuellement depuis l'audit
# =========================================================
ANCRES_ESCALES = {
    "Île de Crespo":                "Ce billet était libellé en ces termes:",
    "Archipel des Pomotou":         "Ce terrible spectacle inaugurait la série des catastrophes maritimes,",
    "Archipel des nouvelles Hébrides": "Le 25 décembre, le _Nautilus_ naviguait au milieu de l'archipel des",
    "La Pouasie":                   "Deux jours après avoir traversé la mer de Corail, le 4 janvier, nous",
    "Cap Wessel":                   "Nous marchions directement vers l'ouest, et, le 11 janvier, nous",
    "Mer de Timor":                 "Le 13 janvier, le capitaine Nemo, arrivé dans la mer de Timor, avait",
    "Île Keeling":                  "Le 24 au matin, par 12° 5′ de latitude sud et 94° 33′ de longitude,",
    "Golfe du Bengale":             "Le 27 janvier, à l'ouvert du vaste golfe du Bengale, nous rencontrâmes",
    "Isthme de Suez":               "Nous avions fait alors seize mille deux cent vingt milles,",
    "L'Archipel grec":             "Le lendemain, 12 février, au lever du jour, le _Nautilus_ remonta à la",
    "Méditerranée":                 "La Méditerranée, la mer bleue par excellence,",
    "L'Atlantide":                 "Le soir, vers onze heures, je reçus la visite très-inattendue du",
    "La mer des Sargasses":         "Toute cette journée du 22 février se passa dans la mer de Sargasses,",
    "Contour Afrique Nord":         "Je demandai au capitaine Nemo s'il avait observé des poissons à des",
    "Cap Horn":                     "Pendant la nuit du 13 au 14 mars, le _Nautilus_ reprit sa direction",
    "Pôle Sud":                     "Le lendemain 19 mars, à cinq heures du matin, je repris mon poste dans",
    "Cap Horn (retour)":            "Le lendemain, premier avril, lorsque le _Nautilus_ remonta à la surface",
    "Contour Amérique Sud (retour)":"Jusqu'au 3 avril, nous ne quittâmes pas les parages de la Patagonie,",
    "Martinique et Guadeloupe":     "Le 20 avril, nous étions remontés à une hauteur moyenne de quinze",
    "Île Lucayes":                  "Cette terrible scène du 20 avril, aucun de nous ne pourra jamais",
    "Cap Hatteras":                 "Le 8 mai, nous étions encore en travers du cap Hatteras, à la",
    "Long Island":                  "La tempête éclata dans la journée du 18 mai, précisément lorsque le",
    "Terre Neuve":                  "Le 15 mai, nous étions sur l'extrémité méridionale du banc de",
    "Irlande":                      "Or, le 25 mai, le _Nautilus_, immergé par trois mille huit cent",
    "Manche":                       "S'il voulait entrer en Manche, il lui fallait prendre franchement à",
    "Contour Angleterre Sud":       "Je revins au salon. Le Nautilus émergeait toujours. Quelques lueurs",
    "Contour Angleterre Nord":      "Puis, une pensée soudaine me terrifia. Le capitaine Nemo avait quitté",
    "Maelström":                    "Le Maelstrom! Un nom plus effrayant dans une situation plus effrayante",
}

# =========================================================
# EXTRACTION DIRECTE — sans IA, 100% fidèle au texte
# =========================================================
def extraire_passage_direct(texte, nom_escale, nb_paragraphes=6):
    """
    Extrait directement les paragraphes depuis le texte brut.
    Aucune IA impliquée — copie exacte de Jules Verne.
    """
    ancre = ANCRES_ESCALES.get(nom_escale)
    if not ancre:
        return None

    # Cherche l'ancre avec longueurs décroissantes
    idx = -1
    for longueur in [40, 30, 20, 15]:
        idx = texte.find(ancre[:longueur])
        if idx != -1:
            break

    if idx == -1:
        return None

    # Extrait le texte à partir de l'ancre
    extrait_brut = texte[idx:idx+7000]

    # Découpe en paragraphes
    paragraphes = [p.strip() for p in extrait_brut.split("\n\n") if p.strip() and len(p.strip()) > 25]

    # Prend les N premiers
    selection = paragraphes[:nb_paragraphes]
    return "\n\n".join(selection)

def vers_html(texte_brut):
    if not texte_brut:
        return "<p><em>Extrait non disponible.</em></p>"
    paragraphes = [p.strip() for p in texte_brut.split("\n\n") if p.strip() and len(p.strip()) > 25]
    resultat = []
    for p in paragraphes:
        # Nettoie les sauts de ligne simples à l'intérieur des paragraphes
        p = re.sub(r"\n", " ", p)
        # Convertit _texte_ en italique
        p = re.sub(r"_([^_]+?)_", lambda m: "<em>" + m.group(1) + "</em>", p)
        resultat.append(f"<p style='margin-bottom:1.5rem;line-height:2.0;font-size:1.25rem;'>{p}</p>")
    return "\n".join(resultat)

# ── Génération des extraits ──
extraits_content = {}
MSG_INTER = "<p><em>Point de navigation intermédiaire — aucun chapitre spécifique dans le roman.</em></p>"

for i, escale in enumerate(data):
    nom_fr  = get_lang(escale.get("Escales du Nautilus"), "fr")
    ch_info = escale.get("chapitre")

    print(f"[{i+1}/{len(data)}] 🧭 {nom_fr}")

    if not ch_info:
        extraits_content[str(i)] = MSG_INTER
        print("  ⏭️  Point intermédiaire")
        continue

    passage = extraire_passage_direct(fr_text_full, nom_fr)
    if passage:
        extraits_content[str(i)] = vers_html(passage)
        print(f"  ✅ Extrait direct ({len(passage)} car.)")
    else:
        extraits_content[str(i)] = "<p><em>Ancre non trouvée pour cette escale.</em></p>"
        print(f"  ⚠️  Ancre non trouvée")

print(f"\n✅ {len(extraits_content)} extraits générés — aucun appel IA")

# ── Charge les espèces maritimes depuis le JSON ──
with open("especes_maritimes_livre.json", "r", encoding="utf-8") as f_esp:
    especes_data = json.load(f_esp)
print(f"🐟 {len(especes_data)} escales chargées")

especes_content = {}
MSG_AUCUNE = "<p class=\'esp-vide\'><em>Aucune espèce maritime identifiée dans ce passage.</em></p>"
for i, escale in enumerate(data):
    nom_fr = get_lang(escale.get("Escales du Nautilus"), "fr")
    esp = especes_data.get(nom_fr, None)
    if not esp or (not esp.get("faune") and not esp.get("flore")):
        especes_content[str(i)] = MSG_AUCUNE
        continue
    html = ""
    if esp.get("faune"):
        html += "<p class=\'esp-titre\'>━━ FAUNE MARITIME ━━</p><ul class=\'esp-liste\'>"
        for e in esp["faune"]:
            html += f"<li>{e}</li>"
        html += "</ul>"
    if esp.get("flore"):
        html += "<p class=\'esp-titre\'>━━ FLORE MARITIME ━━</p><ul class=\'esp-liste\'>"
        for e in esp["flore"]:
            html += f"<li>{e}</li>"
        html += "</ul>"
    especes_content[str(i)] = html
print(f"✅ {len(especes_content)} escales avec espèces formatées")

# =========================================================
# DATAFRAME
# =========================================================
df = pd.DataFrame(data)
df["Latitude_Décimal"]  = pd.to_numeric(df["Latitude_Décimal"],  errors="coerce")
df["Longitude_Décimal"] = pd.to_numeric(df["Longitude_Décimal"], errors="coerce")
df = df.dropna(subset=["Latitude_Décimal", "Longitude_Décimal"]).reset_index(drop=True)
df["DistanceCentreTerre"] = df["Latitude_Décimal"].apply(distance_from_center)
print(f"✅ {len(df)} escales avec coordonnées valides")

# =========================================================
# POPUP HUD
# =========================================================
def make_popup(row, i):
    nom   = html_module.escape(get_lang(row.get("Escales du Nautilus"), "fr") or f"Escale {i+1}")
    date  = html_module.escape(get_lang(row.get("Date"),      "fr"))
    ocean = html_module.escape(get_lang(row.get("Océan/Mer"), "fr"))
    event = html_module.escape(get_lang(row.get("Event"),     "fr"))
    lat   = row["Latitude_Décimal"]
    lon   = row["Longitude_Décimal"]
    dist  = f"{row['DistanceCentreTerre']:,}"

    return f"""
    <div class="hud-popup">
      <div class="hud-lang-label">🧭 Escale du Nautilus</div>
      <div class="hud-body">
        <div class="hud-row"><span class="hud-key">Escale :</span> <span class="hud-val">{nom}</span></div>
        <div class="hud-row"><span class="hud-key">Date :</span> <span class="hud-val">{date}</span></div>
        <div class="hud-row"><span class="hud-key">Océan / Mer :</span> <span class="hud-val">{ocean}</span></div>
        <div class="hud-row"><span class="hud-key">Événement :</span> <span class="hud-val">{event}</span></div>
        <div class="hud-row"><span class="hud-key">Latitude :</span> <span class="hud-val">{lat}</span></div>
        <div class="hud-row"><span class="hud-key">Longitude :</span> <span class="hud-val">{lon}</span></div>
        <div class="hud-row"><span class="hud-key">Distance centre Terre :</span> <span class="hud-val">{dist} m</span></div>
      </div>
      <button class="hud-btn" data-stop="{i}" data-action="book_excerpt">
        📖 Extrait du livre
      </button>
      <button class="hud-btn" data-stop="{i}" data-action="especes" style="margin-top:8px;">
        🐟 Espèces maritimes du roman
      </button>
    </div>"""

print("✅ make_popup HUD prêt")

# =========================================================
# CARTE FOLIUM
# =========================================================
m = folium.Map(location=[20, -30], zoom_start=2, tiles="CartoDB positron")
marker_cluster = MarkerCluster().add_to(m)
coords = []

for i, (_, row) in enumerate(df.iterrows()):
    lat = row["Latitude_Décimal"]
    lon = row["Longitude_Décimal"]
    coords.append([lat, lon])
    tooltip = html_module.escape(get_lang(row.get("Escales du Nautilus"), "fr") or f"Escale {i+1}")

    popup = folium.Popup(make_popup(row, i), max_width=420)
    if i == 0:
        folium.Marker(location=[lat, lon], popup=popup,
            tooltip=f"⚓ Départ — {tooltip}",
            icon=folium.Icon(color="green", icon="anchor", prefix="fa")).add_to(m)
    elif i == len(df) - 1:
        folium.Marker(location=[lat, lon], popup=popup,
            tooltip=f"🏁 Arrivée — {tooltip}",
            icon=folium.Icon(color="red", icon="flag", prefix="fa")).add_to(m)
    else:
        folium.Marker(location=[lat, lon], popup=popup, tooltip=tooltip,
            icon=folium.Icon(color="cadetblue", icon="info-sign")).add_to(marker_cluster)

if len(coords) > 1:
    folium.PolyLine(coords, color="#27c39f", weight=2, opacity=0.7).add_to(m)

print(f"✅ {len(coords)} marqueurs")

# =========================================================
# CSS HUD + MODAL + JS
# =========================================================
custom_html = f"""
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Michroma&display=swap" rel="stylesheet">
<style>
.leaflet-popup-content-wrapper {{
  background: rgba(5,15,25,0.96) !important;
  border: 1px solid rgba(39,195,159,0.30) !important;
  border-radius: 18px !important;
  box-shadow: 0 8px 32px rgba(0,0,0,0.55) !important;
  backdrop-filter: blur(12px);
}}
.leaflet-popup-tip {{ background: rgba(5,15,25,0.96) !important; }}
.leaflet-popup-content {{ margin:0 !important; padding:0 !important; }}
.hud-popup {{
  min-width: 360px; max-width: 420px;
  padding: 20px 22px 16px;
  font-family: 'Raleway', Arial, sans-serif;
}}
.hud-lang-label {{
  font-family: 'Michroma', sans-serif;
  font-size: 0.78rem; letter-spacing: 0.20em;
  text-transform: uppercase; color: #a8ffff;
  margin-bottom: 14px;
  border-bottom: 1px solid rgba(39,195,159,0.18);
  padding-bottom: 8px;
}}
.hud-body {{ display:flex; flex-direction:column; gap:8px; }}
.hud-row {{ font-size: 1.05rem; color:#c8dde8; line-height:1.65; }}
.hud-key {{ color:#27c39f; font-weight:600; margin-right:4px; }}
.hud-val {{ color:#e0f4ff; }}
.hud-btn {{
  margin-top:16px; width:100%;
  padding:10px 14px; background:transparent;
  border:1px solid rgba(39,195,159,0.5);
  border-radius:8px; color:#27c39f;
  font-family:'Michroma',sans-serif;
  font-size:0.82rem; letter-spacing:0.10em;
  cursor:pointer; transition:all 0.2s ease;
  position:relative; overflow:hidden;
}}
.hud-btn:hover {{
  background:rgba(39,195,159,0.12);
  border-color:#27c39f;
  box-shadow:0 0 14px rgba(39,195,159,0.30);
}}
.hud-btn.loading {{ pointer-events:none; color:transparent; }}
.hud-btn.loading::after {{
  content:''; position:absolute; inset:0;
  background:linear-gradient(90deg,rgba(39,195,159,0) 0%,rgba(39,195,159,0.40) 50%,rgba(39,195,159,0) 100%);
  animation:shimmer 1.1s ease-in-out infinite;
}}
@keyframes shimmer {{ 0%{{transform:translateX(-100%)}} 100%{{transform:translateX(100%)}} }}
#hudModal {{
  display:none; position:fixed; inset:0; z-index:9999;
  align-items:center; justify-content:center;
  background:rgba(0,0,0,0.65); backdrop-filter:blur(5px);
}}
#hudModal.open {{ display:flex; }}
#hudModalBox {{
  width:min(820px,92vw); max-height:85vh;
  background:linear-gradient(160deg,rgba(5,18,28,0.98),rgba(3,10,18,0.96));
  border:1px solid rgba(168,255,255,0.22); border-radius:22px;
  box-shadow:0 24px 60px rgba(0,0,0,0.55);
  display:flex; flex-direction:column; overflow:hidden;
}}
#hudModalHeader {{
  padding:16px 24px 14px;
  border-bottom:1px solid rgba(39,195,159,0.18);
  display:flex; align-items:center; justify-content:space-between;
  flex-shrink:0;
}}
#hudModalTitle {{
  font-family:'Michroma',sans-serif;
  font-size:0.88rem; letter-spacing:0.16em;
  color:#27c39f; text-transform:uppercase;
}}
#hudModalClose {{
  background:none; border:none; color:#a8ffff;
  font-size:1.3rem; cursor:pointer; opacity:0.6;
  transition:opacity 0.2s; line-height:1;
}}
#hudModalClose:hover {{ opacity:1; }}
#hudModalBody {{
  padding:28px 34px; overflow-y:auto;
  color:#d5edf5;
  font-family:'Raleway',Arial,sans-serif;
  font-size:1.25rem; line-height:2.0;
  scrollbar-width:thin;
  scrollbar-color:#27c39f rgba(255,255,255,0.06);
}}
#hudModalBody::-webkit-scrollbar {{ width:8px; }}
#hudModalBody::-webkit-scrollbar-track {{ background:rgba(255,255,255,0.04); border-radius:999px; }}
#hudModalBody::-webkit-scrollbar-thumb {{ background:linear-gradient(#27c39f,#195480); border-radius:999px; }}
#hudModalBody p {{ margin-bottom:1.5rem; }}
.esp-titre {{ color:#a8ffff; font-family:\'Michroma\',sans-serif; font-size:0.95rem; letter-spacing:0.15em; margin:1.2rem 0 0.8rem; }}
.esp-liste {{ list-style:none; padding:0; display:grid; gap:0.5rem; }}
.esp-liste li {{ font-size:1.25rem; line-height:1.7; color:#e0f4ff; padding-left:1.2rem; position:relative; }}
.esp-liste li::before {{ content:"•"; position:absolute; left:0; color:#27c39f; }}
.esp-vide {{ color:#a8b8c0 !important; font-size:1.1rem; }}
</style>

<div id="hudModal">
  <div id="hudModalBox">
    <div id="hudModalHeader">
      <span id="hudModalTitle">📖 Journal de bord vivant</span>
      <button id="hudModalClose" onclick="closeHudModal()">✕</button>
    </div>
    <div id="hudModalBody"></div>
  </div>
</div>

<script>
const EXTRAITS = {json.dumps(extraits_content, ensure_ascii=False)};
function openHudModal(content) {{
  document.getElementById("hudModalBody").innerHTML = content;
  document.getElementById("hudModal").classList.add("open");
}}
function closeHudModal() {{
  document.getElementById("hudModal").classList.remove("open");
}}
document.getElementById("hudModal").addEventListener("click", function(e) {{
  if (e.target === this) closeHudModal();
}});
const ESPECES = {json.dumps(especes_content, ensure_ascii=False)};

document.addEventListener("click", function(e) {{
  const btn = e.target.closest(".hud-btn");
  if (!btn) return;
  const stop = btn.getAttribute("data-stop");
  const action = btn.getAttribute("data-action");
  btn.classList.add("loading");
  setTimeout(function() {{
    btn.classList.remove("loading");
    let content = "<p><em>Contenu disponible bientôt.</em></p>";
    let titre = "Journal de bord";
    if (action === "book_excerpt") {{
      content = EXTRAITS[stop] || content;
      titre = "📖 Extrait du livre";
    }} else if (action === "especes") {{
      content = ESPECES[stop] || "<p><em>Aucune espèce maritime identifiée.</em></p>";
      titre = "🐟 Espèces maritimes du roman";
    }}
    document.getElementById("hudModalTitle").innerHTML = titre;
    openHudModal(content);
  }}, 1800);
}});
</script>
"""

m.get_root().html.add_child(folium.Element(custom_html))
m.save(str(OUTPUT_HTML))
print(f"✅ Fichier généré : {OUTPUT_HTML.resolve()}")
display(HTML(f'<a href="{OUTPUT_HTML.name}" target="_blank">🔗 Ouvrir la carte HTML</a>'))    


✅ C:\Users\Utilisateur\Documents\Nautilus_IA_Visualisation\notebooks\data_texts
✅ C:\Users\Utilisateur\Documents\Nautilus_IA_Visualisation\notebooks\data_texts\20000lieues_fr.txt
✅ C:\Users\Utilisateur\Documents\Nautilus_IA_Visualisation\notebooks\Points-escales-chapitres.json
📖 FR : 908,396 caractères
🗺️  32 escales chargées
✅ Utilitaires chargés
[1/32] 🧭 Île de Crespo
  ✅ Extrait direct (591 car.)
[2/32] 🧭 Archipel des Pomotou
  ✅ Extrait direct (2659 car.)
[3/32] 🧭 Archipel des nouvelles Hébrides
  ✅ Extrait direct (1338 car.)
[4/32] 🧭 La Pouasie
  ✅ Extrait direct (2553 car.)
[5/32] 🧭 Cap Wessel
  ✅ Extrait direct (3258 car.)
[6/32] 🧭 Mer de Timor
  ✅ Extrait direct (2830 car.)
[7/32] 🧭 Île Keeling
  ✅ Extrait direct (1897 car.)
[8/32] 🧭 Golfe du Bengale
  ✅ Extrait direct (1642 car.)
[9/32] 🧭 Contour Afrique Est
  ⏭️  Point intermédiaire
[10/32] 🧭 Sous l'isthme de Suez
  ⚠️  Ancre non trouvée
[11/32] 🧭 L'archipel grec
  ⚠️  Ancre non trouvée
[12/32] 🧭 Méditerranée
  ✅ Extrait dire